# 第六次作业：sklearn 逻辑回归源码解析（`LogisticRegression`）

> 主题：阅读 scikit-learn 的实现。

> **主线**：`linear_model.LogisticRegression` 与 `LogisticRegressionCV`；`SGDClassifier(loss="log_loss")` 不在主线（hw5 已讲 SGD 框架）。

环境：`scikit-learn 1.3.2` / Python 3.8。文中行号均以 **1.3.2** 版本的库为准；notebook 内的行号一律由 `inspect` 现场取得。

**`LogisticRegression` 不是「一个算法 + 若干优化器」，而是 6 个 solver 字符串映射到 5 套互不共享代码的后端，外层再叠加 3 种多分类实现与 3 份 log loss 实现。类本身的 Python 代码只做参数校验、数据整形、结果拼装。**

## 0. 环境与源码地图

逻辑回归的实现横跨四层，其中三层在运行时是**编译产物**：

| 文件 | 行数 | 形式 | 内容 |
|---|---|---|---|
| `linear_model/_logistic.py` | 2114 | 纯 Python | `LogisticRegression`（类体 623 行，docstring 284 行）、`LogisticRegressionCV`（698 行）、`_logistic_regression_path` |
| `linear_model/_sag.py` | 372 | 纯 Python | SAG/SAGA 外层：步长估计、warm start 内存、dtype 分派 |
| `linear_model/_linear_loss.py` | 658 | 纯 Python | `LinearModelLoss`：loss / gradient / hessian / hessp |
| `linear_model/_glm/_newton_solver.py` | 525 | 纯 Python | `NewtonSolver` 抽象基类 + `NewtonCholeskySolver` |
| `svm/_base.py` | 1262 | 纯 Python | `_fit_liblinear` 包装 + `_get_liblinear_solver_type` 魔数表 |
| `utils/optimize.py` | 267 | 纯 Python | `_newton_cg` |
| `linear_model/_sag_fast*.so` | — | Cython | SAG/SAGA 内循环 |
| `_loss/_loss*.so` | — | Cython | log loss / sigmoid / softmax 内核 |
| `svm/_liblinear*.so` | — | C++ | liblinear 坐标下降内核 |

因此本文有两条证据线：**能 dump 的 Python 代码**给结构，**不能 dump 的 `.so`** 只给行为实测。凡涉及 liblinear、`_sag_fast`、`_loss` 内循环的结论，都不假装引用源码。

`LogisticRegression` 的类体只有 623 行，其中 284 行是 docstring；扣掉文档字符串后真正的代码约 339 行 —— 它主要在做参数校验、`classes_` 整形、joblib 调度与结果拼装，优化本身全部外包。

In [1]:
import ast
import inspect
import os
import sys

import numpy as np
import sklearn
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

LOGISTIC_PATH = inspect.getsourcefile(LogisticRegression)
LOGISTIC_LINES = open(LOGISTIC_PATH, encoding="utf-8").read().splitlines()


def show_file(path, ranges, title=None):
    # 按行号区间打印源码；行号来自当前安装的 1.3.2 文件
    if title:
        print(title)
    lines = open(path, encoding="utf-8").read().splitlines()
    for a, b in ranges:
        print("  L%d-L%d" % (a, b))
        for i in range(a - 1, b):
            print("%4d| %s" % (i + 1, lines[i].rstrip()))
        print()


def show_lines(ranges, title=None):
    show_file(LOGISTIC_PATH, ranges, title)


print("python   :", sys.version.split()[0])
print("sklearn  :", sklearn.__version__)
print("numpy    :", np.__version__)
print("logistic :", LOGISTIC_PATH)
print()

print("== 类体与 docstring ==")
tree = ast.parse("\n".join(LOGISTIC_LINES))
for name in ["LogisticRegression", "LogisticRegressionCV"]:
    node = next(n for n in tree.body if isinstance(n, ast.ClassDef) and n.name == name)
    doc = node.body[0]
    span = node.end_lineno - node.lineno + 1
    doc_span = doc.end_lineno - doc.lineno + 1
    print("  %-22s 类体 %4d 行（L%d–L%d），docstring %3d 行，其余 %3d 行"
          % (name, span, node.lineno, node.end_lineno, doc_span, span - doc_span))

python   : 3.8.20
sklearn  : 1.3.2
numpy    : 1.24.4
logistic : /root/MSA/ENTER/envs/vmunet/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py

== 类体与 docstring ==
  LogisticRegression     类体  623 行（L792–L1414），docstring 284 行，其余 339 行
  LogisticRegressionCV   类体  698 行（L1417–L2114），docstring 277 行，其余 421 行


In [2]:
import sklearn.linear_model._logistic as M_LOGISTIC
import sklearn.linear_model._sag as M_SAG
import sklearn.linear_model._linear_loss as M_LOSS_PY
import sklearn.linear_model._glm._newton_solver as M_NEWTON
import sklearn.svm._base as M_SVM_BASE
import sklearn.utils.optimize as M_OPT
import sklearn.linear_model._sag_fast as M_SAG_FAST
import sklearn._loss._loss as M_LOSS_SO
import sklearn.svm._liblinear as M_LIBLINEAR

print("== 纯 Python（可以 inspect）==")
for name, mod in [("linear_model/_logistic", M_LOGISTIC),
                  ("linear_model/_sag", M_SAG),
                  ("linear_model/_linear_loss", M_LOSS_PY),
                  ("linear_model/_glm/_newton_solver", M_NEWTON),
                  ("svm/_base", M_SVM_BASE),
                  ("utils/optimize", M_OPT)]:
    n = len(open(mod.__file__, encoding="utf-8").read().splitlines())
    print("  %-36s %5d 行" % (name, n))

print()
print("== 编译产物（不能 inspect）==")
for name, mod in [("linear_model/_sag_fast", M_SAG_FAST),
                  ("_loss/_loss", M_LOSS_SO),
                  ("svm/_liblinear", M_LIBLINEAR)]:
    print("  %-36s %s" % (name, os.path.basename(mod.__file__)))
    try:
        inspect.getsource(mod)
    except (OSError, TypeError) as e:
        print("      inspect.getsource -> %s" % type(e).__name__)

== 纯 Python（可以 inspect）==
  linear_model/_logistic                2114 行
  linear_model/_sag                      372 行
  linear_model/_linear_loss              658 行
  linear_model/_glm/_newton_solver       525 行
  svm/_base                             1262 行
  utils/optimize                         267 行

== 编译产物（不能 inspect）==
  linear_model/_sag_fast               _sag_fast.cpython-38-x86_64-linux-gnu.so
      inspect.getsource -> OSError
  _loss/_loss                          _loss.cpython-38-x86_64-linux-gnu.so
      inspect.getsource -> OSError
  svm/_liblinear                       _liblinear.cpython-38-x86_64-linux-gnu.so
      inspect.getsource -> OSError


## 1. 六个 solver 字符串 = 五套互不共享代码的后端

`_check_solver` 只做合法性检查，不选后端。真正的分派发生在两个地方：

1. **`fit` 里 liblinear 直接早退**：`solver == "liblinear"` 时调用 `_fit_liblinear` 后立刻 `return self`，**完全不经过** `_logistic_regression_path`；
2. **path 里的四支**：其余五个 solver 在同一个 `for C in Cs` 循环内分成 `lbfgs` / `newton-cg` / `newton-cholesky` / `sag+saga` 四支。

| solver | 后端 | 入口 | `n_iter_` 来源 |
|---|---|---|---|
| `liblinear` | liblinear C++（坐标下降） | `fit` → `_fit_liblinear` | C++ 返回的每类轮数 |
| `lbfgs` | `scipy.optimize.minimize(method="L-BFGS-B")` | path | `opt_res.nit` |
| `newton-cg` | sklearn 自写 `_newton_cg` | path | 共轭梯度步数 |
| `newton-cholesky` | `NewtonCholeskySolver` | path | `sol.iteration` |
| `sag` / `saga` | Cython `_sag_fast` | `_sag.py::sag_solver` | 整轮 epoch |

`solver` 不只是「优化器开关」，它同时决定：能否 `inspect.getsource`、支持什么 `penalty`、`tol` 的量纲、`n_iter_` 的语义、并行方式（进程 / 线程 / 无）。

还有一个结构细节：path 里其实也有一段 `elif solver == "liblinear"`，但 `LogisticRegression.fit` 永远走不到它 —— 它只服务 `LogisticRegressionCV`（CV 通过 `_log_reg_scoring_path` 调 path）。

**同解实测**：同一份 300×8 二分类数据，`tol=1e-6, max_iter=1000`。六个 solver 的系数差异都在 `5.5e-4` 以内，但目标函数值不同：liblinear 是 `111.928882`，其余是 `111.928737`。这是**优化精度**的差异，不是目标函数不同。

In [3]:
from sklearn.linear_model._logistic import (
    _check_multi_class,
    _check_solver,
    _logistic_regression_path,
)
from sklearn.svm._base import _fit_liblinear, _get_liblinear_solver_type

print("== _logistic_regression_path 的 solver 分支 ==")
src, start = inspect.getsourcelines(_logistic_regression_path)
loop = next(i for i, l in enumerate(src) if "for i, C in enumerate(Cs):" in l)
for i, line in enumerate(src):
    if i < loop:
        continue
    s = line.strip()
    if s in ('if solver == "lbfgs":', 'elif solver == "newton-cg":',
             'elif solver == "newton-cholesky":', 'elif solver == "liblinear":',
             'elif solver in ["sag", "saga"]:'):
        print("  L%-5d %s" % (start + i, s))

print()
print("== LogisticRegression.fit 里的 liblinear 早退 ==")
fit_src, fit_start = inspect.getsourcelines(LogisticRegression.fit)
for i, line in enumerate(fit_src):
    if line.strip() == 'if solver == "liblinear":':
        print("  L%-5d %s" % (fit_start + i, line.strip()))

print()
print("== liblinear 的两个入口 ==")
print("  _fit_liblinear              L%d" % inspect.getsourcelines(_fit_liblinear)[1])
print("  _get_liblinear_solver_type  L%d"
      % inspect.getsourcelines(_get_liblinear_solver_type)[1])

== _logistic_regression_path 的 solver 分支 ==
  L447   if solver == "lbfgs":
  L467   elif solver == "newton-cg":
  L473   elif solver == "newton-cholesky":
  L488   elif solver == "liblinear":
  L516   elif solver in ["sag", "saga"]:

== LogisticRegression.fit 里的 liblinear 早退 ==
  L1221  if solver == "liblinear":

== liblinear 的两个入口 ==
  _fit_liblinear              L1066
  _get_liblinear_solver_type  L1009


In [4]:
import warnings

from sklearn.datasets import make_classification
from sklearn.linear_model._linear_loss import LinearModelLoss
from sklearn._loss.loss import HalfBinomialLoss
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=300, n_features=8, n_informative=5, random_state=0)
Xs = StandardScaler().fit_transform(X)
yf = y.astype(np.float64)
sw = np.ones(len(y))

solvers = ["lbfgs", "liblinear", "newton-cg", "newton-cholesky", "sag", "saga"]
loss = LinearModelLoss(base_loss=HalfBinomialLoss(), fit_intercept=True)

print("%-15s %-11s %10s %14s %6s"
      % ("solver", "n_iter_", "|dcoef|", "objective", "warn"))
base = None
for s in solvers:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        m = LogisticRegression(solver=s, tol=1e-6, max_iter=1000,
                               random_state=0).fit(Xs, y)
    if base is None:
        base = m.coef_.copy()
    w = np.concatenate([m.coef_.ravel(), m.intercept_]).astype(np.float64)
    obj = loss.loss(w, Xs, yf, sample_weight=sw, l2_reg_strength=1.0)
    print("%-15s %-11s %10.2e %14.9f %6d"
          % (s, str(m.n_iter_.tolist()), np.abs(m.coef_ - base).max(),
             obj, len(caught)))

solver          n_iter_        |dcoef|      objective   warn
lbfgs           [10]          0.00e+00  111.928737107      0
liblinear       [7]           5.51e-04  111.928882173      0
newton-cg       [10]          2.42e-05  111.928737089      0
newton-cholesky [5]           2.57e-05  111.928737089      0
sag             [36]          2.43e-05  111.928737089      0
saga            [22]          2.30e-05  111.928737089      0


## 2. `C`、`penalty=None`、`sample_weight`：同一个目标函数的三种尺度

统一写成一个未归一化目标：

```text
sum_i sample_weight_i * loss_i(w) + ||w||^2 / (2C)
```

但进入后端前，这个式子被改写成了三种不同尺度：

| 后端 | 实际目标 | 缩放方式 |
|---|---|---|
| `lbfgs` / `newton-cg` | `LinearModelLoss` + `l2_reg_strength = 1/C` | 直接 |
| `newton-cholesky` | 先把 `sample_weight` 归一到和为 1，再令 `l2_reg_strength = 1/(C * sum(sw))` | 源码注释原话 `nasty matter` |
| `sag` / `saga` | `alpha = 1/C`，再 `alpha_scaled = alpha / n_samples` | 因为 SAG 的目标是 `(1/n) sum loss` |
| `liblinear` | C++ 内部 `0.5||w||^2 + C * sum loss` | 整体除以 C |

`penalty=None` 不是一条独立的「无正则」分支：`fit` 把 `C_` 换成 `np.inf`，再把 `penalty` 改回 `"l2"`。因此 `C` 被完全忽略，而且**只有 `C != 1` 时才发警告**。

liblinear 更特殊：`_check_solver` 只拦字符串 `"none"`，`None` 会一路传到 `_get_liblinear_solver_type`，最后抛查表式错误。同一个「关正则」，两种写法、两套错误路径。

`sample_weight` 整体缩放会改变模型：因为权重与 `C` 在目标函数里是耦合的。把权重全乘 7，等价于把 `C` 乘 7。

In [5]:
from sklearn.linear_model._sag import sag_solver

show_lines([(300, 315), (476, 476), (522, 528), (1191, 1201)],
           title="== 目标函数的三种尺度（_logistic.py）==")
show_file(inspect.getsourcefile(sag_solver), [(254, 257)],
          title="== SAG 的 alpha 再除以 n_samples（_sag.py）==")

== 目标函数的三种尺度（_logistic.py）==
  L300-L315
 300|     if solver == "newton-cholesky":
 301|         # IMPORTANT NOTE: Rescaling of sample_weight:
 302|         # Same as in _GeneralizedLinearRegressor.fit().
 303|         # We want to minimize
 304|         #     obj = 1/(2*sum(sample_weight)) * sum(sample_weight * deviance)
 305|         #         + 1/2 * alpha * L2,
 306|         # with
 307|         #     deviance = 2 * log_loss.
 308|         # The objective is invariant to multiplying sample_weight by a constant. We
 309|         # choose this constant such that sum(sample_weight) = 1. Thus, we end up with
 310|         #     obj = sum(sample_weight * loss) + 1/2 * alpha * L2.
 311|         # Note that LinearModelLoss.loss() computes sum(sample_weight * loss).
 312|         #
 313|         # This rescaling has to be done before multiplying by class_weights.
 314|         sw_sum = sample_weight.sum()  # needed to rescale penalty, nasty matter!
 315|         sample_weight = sample_weig

In [6]:
print("== penalty=None：C 被换成 inf ==")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    a = LogisticRegression(penalty=None, C=1).fit(X, y)
    b = LogisticRegression(penalty=None, C=2).fit(X, y)
msg = [str(w.message) for w in caught if "ignore" in str(w.message)]
print("  C=1 vs C=2 系数逐元素相同 :",
      np.array_equal(a.coef_, b.coef_) and np.array_equal(a.intercept_, b.intercept_))
print("  C=2 时的警告             :", msg)
print()

print("== liblinear 的两种写法 ==")
for pen in [None, "none"]:
    try:
        LogisticRegression(penalty=pen, solver="liblinear").fit(X, y)
    except ValueError as e:
        print("  penalty=%-6r -> %s" % (pen, str(e).split(", Parameters")[0]))
print()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    LogisticRegression(penalty="none").fit(X, y)
print("  字符串 'none' 的额外警告 :",
      [str(w.message) for w in caught if "deprecated" in str(w.message)])

== penalty=None：C 被换成 inf ==
  C=1 vs C=2 系数逐元素相同 : True
  C=2 时的警告             : ['Setting penalty=None will ignore the C and l1_ratio parameters']

== liblinear 的两种写法 ==
  penalty=None   -> Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported
  penalty='none' -> penalty='none' is not supported for the liblinear solver

  字符串 'none' 的额外警告 : ["`penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`."]


In [7]:
print("== sample_weight 整体缩放 ==")
for s in ["lbfgs", "newton-cholesky"]:
    m1 = LogisticRegression(solver=s, C=0.5, tol=1e-10,
                            max_iter=5000).fit(Xs, y)
    m7 = LogisticRegression(solver=s, C=0.5, tol=1e-10, max_iter=5000).fit(
        Xs, y, sample_weight=np.full(len(y), 7.0))
    eq = LogisticRegression(solver=s, C=3.5, tol=1e-10,
                            max_iter=5000).fit(Xs, y)
    print("  %-15s 权重乘 7：|dcoef|=%.3e" % (s, np.abs(m7.coef_ - m1.coef_).max()))
    print("  %-15s sw=7,C=0.5 vs sw=1,C=3.5：|dcoef|=%.3e"
          % (s, np.abs(m7.coef_ - eq.coef_).max()))

== sample_weight 整体缩放 ==
  lbfgs           权重乘 7：|dcoef|=1.228e-01
  lbfgs           sw=7,C=0.5 vs sw=1,C=3.5：|dcoef|=2.776e-16
  newton-cholesky 权重乘 7：|dcoef|=1.228e-01
  newton-cholesky sw=7,C=0.5 vs sw=1,C=3.5：|dcoef|=0.000e+00


## 3. `n_iter_` 有四种含义，而且 docstring 对 liblinear 写错了

`fit` 把各路径的计数统一成 `np.int32` 数组（liblinear 由 C++ 返回，其余由 `_logistic_regression_path` 汇总），但里面装的是四种不同量纲：

- `lbfgs`：L-BFGS 迭代数；
- `newton-cg`：共轭梯度步数；
- `newton-cholesky`：牛顿迭代数；
- `sag` / `saga`：整轮 epoch；
- `liblinear`：坐标下降轮数。

形状也不是简单的「按类」：二分类恒为 `(1,)`；OvR 多分类为 `(n_classes,)`；multinomial 为 `(1,)`。

docstring 写的是「liblinear 只给所有类的最大迭代数」，1.3.2 的实现却返回**每类一个值**。digits 上实测是 10 个值，不是 1 个。这类「属性名相同、语义不同、文档还落后于实现」的地方，跨 solver 直接比较 `n_iter_` 是没有意义的。

In [8]:
doc_lines = inspect.getdoc(LogisticRegression).splitlines()
i0 = next(i for i, l in enumerate(doc_lines) if l.startswith("n_iter_ :"))
print("== docstring 里的 n_iter_ ==")
for l in doc_lines[i0:i0 + 9]:
    print("  " + l)

== docstring 里的 n_iter_ ==
  n_iter_ : ndarray of shape (n_classes,) or (1, )
      Actual number of iterations for all classes. If binary or multinomial,
      it returns only 1 element. For liblinear solver, only the maximum
      number of iteration across all classes is given.
  
      .. versionchanged:: 0.20
  
          In SciPy <= 1.0.0 the number of lbfgs iterations may exceed
          ``max_iter``. ``n_iter_`` will now report at most ``max_iter``.


In [9]:
from sklearn.datasets import load_digits

Xdig, ydig = load_digits(return_X_y=True)
print("digits:", Xdig.shape, "classes:", np.unique(ydig))
print()

configs = [
    dict(solver="lbfgs", multi_class="ovr", max_iter=300),
    dict(solver="lbfgs", multi_class="multinomial", max_iter=300),
    dict(solver="liblinear", max_iter=200, random_state=0),
    dict(solver="newton-cholesky", max_iter=200),
]
print("== 同一数据、不同 solver / 多分类策略 ==")
for kw in configs:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        m = LogisticRegression(**kw).fit(Xdig, ydig)
    print("  %-14s multi_class=%-12s coef_=%-9s n_iter_ 形状=%-6s 总和=%4d  warn=%d"
          % (kw["solver"], kw.get("multi_class", "auto"), m.coef_.shape,
             m.n_iter_.shape, m.n_iter_.sum(), len(caught)))
    print("      n_iter_ =", m.n_iter_.tolist(), " dtype:", m.n_iter_.dtype)

digits: (1797, 64) classes: [0 1 2 3 4 5 6 7 8 9]

== 同一数据、不同 solver / 多分类策略 ==


  lbfgs          multi_class=ovr          coef_=(10, 64)  n_iter_ 形状=(10,)  总和=2707  warn=6
      n_iter_ = [152, 300, 213, 300, 262, 300, 280, 300, 300, 300]  dtype: int32


  lbfgs          multi_class=multinomial  coef_=(10, 64)  n_iter_ 形状=(1,)   总和= 300  warn=1
      n_iter_ = [300]  dtype: int32
  liblinear      multi_class=auto         coef_=(10, 64)  n_iter_ 形状=(10,)  总和= 130  warn=0
      n_iter_ = [11, 14, 12, 13, 12, 13, 12, 12, 17, 14]  dtype: int32


  newton-cholesky multi_class=auto         coef_=(10, 64)  n_iter_ 形状=(10,)  总和= 110  warn=0
      n_iter_ = [11, 11, 11, 11, 11, 11, 12, 12, 9, 11]  dtype: int32


## 4. 多分类是三种实现，而不是一个 `multi_class` 开关

| 路径 | 谁做 OvR | 代码位置 | `n_iter_` |
|---|---|---|---|
| `liblinear` | **C++ 内部**做 OvR | `fit` 早退 → `_fit_liblinear` | `(n_classes,)` |
| joblib OvR | Python 层按类并行 | `Parallel(...)` + 每类一次 path | `(n_classes,)` |
| multinomial | 一条联合优化 | `classes_ = [None]`，让 joblib 只迭代一次 | `(1,)` |

`multi_class="auto"` 不是「自动挑最好的」，而是一张**能力表**：`liblinear` / `newton-cholesky` 强制 `ovr`；其余 solver 在 `n_classes > 2` 时给 `multinomial`，二分类回 `ovr`。显式写 `multinomial` 遇到不支持的后端直接 `ValueError`。

一个容易误读的细节：multinomial 分支里的 `classes_ = [None]` 不是为了标签，而是为了**让 joblib 的循环只跑一次**；`pos_class` 在 multinomial 路径里根本不参与计算。

并行方式也随 solver 变：`sag` / `saga` 释放 GIL，所以 `prefer="threads"`；其余 solver 用 `prefer="processes"`。

In [10]:
show_lines([(78, 93), (1273, 1278), (1280, 1288)],
           title="== 多分类能力表与 joblib hack ==")

== 多分类能力表与 joblib hack ==
  L78-L93
  78| def _check_multi_class(multi_class, solver, n_classes):
  79|     """Computes the multi class type, either "multinomial" or "ovr".
  80| 
  81|     For `n_classes` > 2 and a solver that supports it, returns "multinomial".
  82|     For all other cases, in particular binary classification, return "ovr".
  83|     """
  84|     if multi_class == "auto":
  85|         if solver in ("liblinear", "newton-cholesky"):
  86|             multi_class = "ovr"
  87|         elif n_classes > 2:
  88|             multi_class = "multinomial"
  89|         else:
  90|             multi_class = "ovr"
  91|     if multi_class == "multinomial" and solver in ("liblinear", "newton-cholesky"):
  92|         raise ValueError("Solver %s does not support a multinomial backend." % solver)
  93|     return multi_class

  L1273-L1278
1273|         # Hack so that we iterate only once for the multinomial case.
1274|         if multi_class == "multinomial":
1275|            

In [11]:
print("== digits：三条多分类路径 ==")
print("%-11s %-14s %-10s %-13s %8s %6s"
      % ("solver", "multi_class", "coef_", "n_iter_ 形状", "acc", "warn"))
fitted = []
for solver, mc in [("liblinear", "auto"), ("lbfgs", "ovr"),
                   ("lbfgs", "multinomial")]:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        m = LogisticRegression(solver=solver, multi_class=mc, max_iter=300,
                               random_state=0).fit(Xdig, ydig)
    fitted.append((solver, mc, m))
    print("%-11s %-14s %-10s %-13s %8.4f %6d"
          % (solver, mc, str(m.coef_.shape), str(m.n_iter_.shape),
             m.score(Xdig, ydig), len(caught)))
    print("    n_iter_ =", m.n_iter_.tolist())

print()
print("== 预测输出的行和 ==")
for solver, mc, m in fitted:
    p = m.predict_proba(Xdig[:3])
    print("  %-11s %-14s %s" % (solver, mc, p.sum(axis=1)))

== digits：三条多分类路径 ==
solver      multi_class    coef_      n_iter_ 形状         acc   warn
liblinear   auto           (10, 64)   (10,)           0.9933      0
    n_iter_ = [11, 14, 12, 13, 12, 13, 12, 12, 17, 14]


lbfgs       ovr            (10, 64)   (10,)           0.9978      6
    n_iter_ = [152, 300, 213, 300, 262, 300, 280, 300, 300, 300]


lbfgs       multinomial    (10, 64)   (1,)            1.0000      1
    n_iter_ = [300]

== 预测输出的行和 ==
  liblinear   auto           [1. 1. 1.]
  lbfgs       ovr            [1. 1. 1.]
  lbfgs       multinomial    [1. 1. 1.]


In [12]:
print("== 不支持的后端组合 ==")
try:
    LogisticRegression(solver="newton-cholesky",
                       multi_class="multinomial").fit(Xdig, ydig)
except ValueError as e:
    print("  newton-cholesky + multinomial ->", e)

try:
    LogisticRegression(solver="liblinear",
                       multi_class="multinomial").fit(Xdig, ydig)
except ValueError as e:
    print("  liblinear + multinomial      ->", e)

== 不支持的后端组合 ==
  newton-cholesky + multinomial -> Solver newton-cholesky does not support a multinomial backend.
  liblinear + multinomial      -> Solver liblinear does not support a multinomial backend.


## 5. 二分类 multinomial 只保存一行系数：半参数化

softmax 对「所有类分数加同一常数」不变，所以 K 类问题的参数只有 K−1 组可辨识。sklearn 的规范是：**第 0 类分数恒为 0**，只保存第 1 行。

`_logistic_regression_path` 在 multinomial 且 `n_classes == 2` 时执行：

```python
multi_w0 = multi_w0[1][np.newaxis, :]
```

丢掉第 0 行。预测时再把这一行「镜像」成两类分数：

```python
decision_2d = np.c_[-decision, decision]
softmax(decision_2d)
```

因此 `coef_.shape == (1, n_features)`，`decision_function` 返回 1D —— 形状与 OvR 完全一样，但两个模型的系数不可比，因为它们优化的是不同损失（OvR 是 logistic，multinomial 是 softmax 交叉熵）。

In [13]:
from scipy.special import expit
from sklearn.utils.extmath import softmax

ovr = LogisticRegression(solver="lbfgs", multi_class="ovr").fit(Xs, y)
mul = LogisticRegression(solver="lbfgs", multi_class="multinomial").fit(Xs, y)

print("== 二分类：两种模式的形状 ==")
for name, m in [("ovr", ovr), ("multinomial", mul)]:
    print("  %-12s coef_=%-8s intercept_=%-8s decision=%s proba=%s"
          % (name, m.coef_.shape, m.intercept_.shape,
             m.decision_function(Xs[:5]).shape, m.predict_proba(Xs[:5]).shape))

print("  multinomial 的 proba 行和:", mul.predict_proba(Xs[:5]).sum(axis=1))
print("  两种模型的 coef_ 最大差: %.3f" % np.abs(ovr.coef_ - mul.coef_).max())
print()

d = mul.decision_function(Xs)
p = mul.predict_proba(Xs)
print("  expit(2d) == predict_proba[:, 1]:", np.allclose(expit(2 * d), p[:, 1]))

w, b = mul.coef_[0], mul.intercept_[0]
two_row = np.vstack([np.zeros_like(w), w])          # 第 0 类固定为 0
scores = np.c_[-(Xs @ w + b), Xs @ w + b]
p2 = softmax(scores)                                # 与源码 predict_proba 同一做法
print("  手工两行系数 + softmax 与 predict_proba 一致:", np.allclose(p2, p))
print("  第一行系数全为 0:", np.all(two_row[0] == 0))

== 二分类：两种模式的形状 ==
  ovr          coef_=(1, 8)   intercept_=(1,)     decision=(5,) proba=(5, 2)
  multinomial  coef_=(1, 8)   intercept_=(1,)     decision=(5,) proba=(5, 2)
  multinomial 的 proba 行和: [1. 1. 1. 1. 1.]
  两种模型的 coef_ 最大差: 0.646

  expit(2d) == predict_proba[:, 1]: True
  手工两行系数 + softmax 与 predict_proba 一致: True
  第一行系数全为 0: True


## 6. 两种牛顿法：matrix-free 的 Hessian-vector product vs 显式 Hessian + Cholesky

两个 solver 共用 `LinearModelLoss`，但取二阶信息的方式相反：

| | `newton-cg` | `newton-cholesky` |
|---|---|---|
| 二阶对象 | `gradient_hessian_product` 返回 `(grad, hessp)` | `gradient_hessian` 填满 `(d, d)` 矩阵 |
| 线性求解 | 共轭梯度，只需 `H @ v` | `scipy.linalg.solve(..., assume_a="sym")` |
| 内存 | `O(d)` | `O(d^2)` |
| `n_iter_` | CG 步数 | 牛顿迭代数 |

`NewtonSolver.solve` 的四步循环对所有牛顿法都一样：`update_gradient_hessian → inner_solve → line_search → check_convergence`。差别只在 `inner_solve`。`newton-cholesky` 在 Hessian 出现大量负值时还会退化成 L-BFGS。

`hessp` 并不是近似：实测 `hessp(v)` 与显式 `H @ v` 的差在 `1e-14` 量级。换句话说，两种求解器的**目标完全相同**，只是内存换时间。

In [14]:
from sklearn._loss.loss import HalfBinomialLoss

m = LogisticRegression(solver="lbfgs").fit(Xs, y)
w = np.concatenate([m.coef_.ravel(), m.intercept_]).astype(np.float64)
loss = LinearModelLoss(base_loss=HalfBinomialLoss(), fit_intercept=True)

g, H, hessian_warning = loss.gradient_hessian(w, Xs, yf, sw, 1.0)
g2, hessp = loss.gradient_hessian_product(w, Xs, yf, sw, 1.0)
v = np.random.RandomState(0).randn(w.size)

print("== 二阶信息的两条路径 ==")
print("  gradient_hessian         -> grad %s, Hessian %s, warning=%s"
      % (g.shape, H.shape, hessian_warning))
print("  gradient_hessian_product -> grad %s, 第二项 callable=%s"
      % (g2.shape, callable(hessp)))
print("  ||hessp(v) - H @ v||_inf = %.2e" % np.abs(hessp(v) - H @ v).max())
print("  Hessian 大小: %d 字节（d=%d）；d=1000 时 %.1f MB；d=10000 时 %.0f MB"
      % (H.nbytes, w.size, 1000 ** 2 * 8 / 1e6, 10000 ** 2 * 8 / 1e6))

== 二阶信息的两条路径 ==
  gradient_hessian         -> grad (9,), Hessian (9, 9), warning=False
  gradient_hessian_product -> grad (9,), 第二项 callable=True
  ||hessp(v) - H @ v||_inf = 2.84e-14
  Hessian 大小: 648 字节（d=9）；d=1000 时 8.0 MB；d=10000 时 800 MB


In [15]:
import time

print("== digits 前 300 个样本：两种牛顿法 ==")
Xsub, ysub = Xdig[:300], ydig[:300]
for s in ["newton-cg", "newton-cholesky"]:
    t0 = time.time()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        m = LogisticRegression(solver=s, max_iter=100).fit(Xsub, ysub)
    mc = _check_multi_class("auto", s, len(np.unique(ysub)))
    print("  %-16s multi_class=%-11s n_iter_=%-18s 形状=%-6s 用时=%.2fs"
          % (s, mc, str(m.n_iter_.tolist()), str(m.n_iter_.shape),
             time.time() - t0))

== digits 前 300 个样本：两种牛顿法 ==
  newton-cg        multi_class=multinomial n_iter_=[23]               形状=(1,)   用时=0.05s


  newton-cholesky  multi_class=ovr         n_iter_=[11, 11, 10, 11, 11, 10, 11, 11, 10, 10] 形状=(10,)  用时=2.66s


## 7. `LogisticRegressionCV`：固定网格 + 每条路径独立求解

`LogisticRegressionCV` 没有「先粗后细」的自适应搜索。`Cs` 是整数时直接执行：

```python
Cs = np.logspace(-4, 4, Cs)
```

与数据、与 `n_samples` 都无关；默认 10 个点，跨度 8 个数量级。

工作方式是三层循环：`n_classes × n_folds × n_cs`。每个 fold 调一次 `_log_reg_scoring_path`，后者一次算完整个 C 网格；它内部再调 `_logistic_regression_path`，后者在 `for C in Cs` 循环里把上一个 C 的解 `w0` 直接当下一个 C 的初值，所以这条路径天然是 warm start。

这也解释了 `n_iter_` 的三维形状：multinomial 是 `(1, n_folds, n_cs)`；OvR 是 `(n_classes, n_folds, n_cs)`。二分类的 `C_` 仍然是长度为 1 的数组，不是标量。

In [16]:
print("== LogisticRegressionCV：默认 C 网格 ==")
cv = LogisticRegressionCV(cv=3, solver="lbfgs", multi_class="multinomial",
                          max_iter=1000).fit(Xs, y)
print("  Cs_ =", np.array2string(cv.Cs_, precision=4))
print("  与 np.logspace(-4, 4, 10) 完全相同:",
      np.allclose(cv.Cs_, np.logspace(-4, 4, 10)))
print("  选中的 C_ =", cv.C_, " coef_ =", cv.coef_.shape)
print("  scores_[1] =", cv.scores_[1].shape,
      " coefs_paths_[1] =", cv.coefs_paths_[1].shape)
print("  n_iter_ =", cv.n_iter_.shape)
print(cv.n_iter_)

== LogisticRegressionCV：默认 C 网格 ==
  Cs_ = [1.0000e-04 7.7426e-04 5.9948e-03 4.6416e-02 3.5938e-01 2.7826e+00
 2.1544e+01 1.6681e+02 1.2915e+03 1.0000e+04]
  与 np.logspace(-4, 4, 10) 完全相同: True
  选中的 C_ = [2.7825594]  coef_ = (1, 8)
  scores_[1] = (3, 10)  coefs_paths_[1] = (3, 10, 9)
  n_iter_ = (1, 3, 10)
[[[ 3  9  9  8  8  8  8  6  2  1]
  [ 3  7  8  9  8  8  6  6  3  1]
  [ 3  8  9 10  9  7  6  6  3  1]]]


## 8. 静默忽略 / 警告 / 报错：参数的实际落点

同一个参数名，在不同 solver 下的命运可能完全不同：

| 参数 | 条件 | 行为 |
|---|---|---|
| `C` | `penalty=None` | 被换成 `inf`，**静默失效**（`C != 1` 时才警告） |
| `l1_ratio` | 非 `elasticnet` | `UserWarning`，不是静默 |
| `intercept_scaling` | 非 `liblinear` | **静默忽略** |
| `warm_start` | `liblinear` | `fit` 早退，**静默忽略** |
| `n_jobs` | `liblinear` 且 > 1 | 警告「不起作用」 |
| `multi_class` | `liblinear` + 显式 `multinomial` | `ValueError`（尽管 docstring 写「ignored」） |
| `dual` | 非 `liblinear` | `ValueError` |
| `tol` | 各后端 | 量纲不同：`gtol` / CG 相对残差 / liblinear 的 `eps` |

liblinear 还有一张**魔数表**：`(multi_class, penalty, loss, dual)` 四元组映射到一个整数，C++ 内核按这个整数选算法。逻辑回归只用到 3 个值：`l2` 原生 `0`、`l2` 对偶 `7`、`l1` 原生 `6`。`penalty=None` 之所以报「Unsupported set of arguments」，就是因为它在这张表里查不到。

In [17]:
from sklearn.svm._base import _get_liblinear_solver_type

src, start = inspect.getsourcelines(_get_liblinear_solver_type)
i0 = next(i for i, l in enumerate(src) if "_solver_type_dict = {" in l)
i1 = next(i for i, l in enumerate(src) if "Unsupported set of arguments" in l)

print("== liblinear 魔数表（从函数首行 L%d 起算）==" % start)
for i in range(i0, i0 + 8):
    print("%4d| %s" % (start + i, src[i].rstrip()))
print("   ...")
for i in range(i1 - 2, i1 + 3):
    print("%4d| %s" % (start + i, src[i].rstrip()))
print()

print("== 逻辑回归用到的三个魔数 ==")
for penalty, dual in [("l2", False), ("l2", True), ("l1", False)]:
    print("  penalty=%-2s dual=%-5s -> solver_type=%d"
          % (penalty, dual,
             _get_liblinear_solver_type("ovr", penalty, "logistic_regression", dual)))

== liblinear 魔数表（从函数首行 L1009 起算）==
1025|     _solver_type_dict = {
1026|         "logistic_regression": {"l1": {False: 6}, "l2": {False: 0, True: 7}},
1027|         "hinge": {"l2": {True: 3}},
1028|         "squared_hinge": {"l1": {False: 5}, "l2": {False: 2, True: 1}},
1029|         "epsilon_insensitive": {"l2": {True: 13}},
1030|         "squared_epsilon_insensitive": {"l2": {False: 11, True: 12}},
1031|         "crammer_singer": 4,
1032|     }
   ...
1059|                 return solver_num
1060|     raise ValueError(
1061|         "Unsupported set of arguments: %s, Parameters: penalty=%r, loss=%r, dual=%r"
1062|         % (error_string, penalty, loss, dual)
1063|     )

== 逻辑回归用到的三个魔数 ==
  penalty=l2 dual=False -> solver_type=0
  penalty=l2 dual=True  -> solver_type=7
  penalty=l1 dual=False -> solver_type=6


In [18]:
print("== 被忽略 / 被警告 / 被报错的参数 ==")

a = LogisticRegression(solver="liblinear", intercept_scaling=1).fit(Xs, y)
b = LogisticRegression(solver="liblinear", intercept_scaling=100).fit(Xs, y)
print("  intercept_scaling(liblinear)  |dcoef|=%.2e"
      % np.abs(a.coef_ - b.coef_).max())
a = LogisticRegression(solver="lbfgs", intercept_scaling=1).fit(Xs, y)
b = LogisticRegression(solver="lbfgs", intercept_scaling=100).fit(Xs, y)
print("  intercept_scaling(lbfgs)      |dcoef|=%.2e  （静默忽略）"
      % np.abs(a.coef_ - b.coef_).max())

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    LogisticRegression(solver="saga", l1_ratio=0.9, max_iter=2000,
                       random_state=0).fit(Xs, y)
print("  l1_ratio + l2                 ->",
      [str(w.message) for w in caught if "l1_ratio" in str(w.message)])

a = LogisticRegression(solver="liblinear", warm_start=True).fit(Xs, y)
b = LogisticRegression(solver="liblinear", warm_start=True).fit(Xs, y)
print("  warm_start(liblinear) 二次拟合 |dcoef|=%.2e  （静默忽略）"
      % np.abs(a.coef_ - b.coef_).max())

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    LogisticRegression(solver="liblinear", n_jobs=2).fit(Xs, y)
print("  n_jobs=2 + liblinear          ->",
      [str(w.message)[:70] for w in caught])
print()

print("== _check_solver 能提前拦下的 penalty ==")
for s in solvers:
    accepted = []
    for p in ["l2", "l1", "elasticnet", None]:
        try:
            _check_solver(s, p, False)
            accepted.append(p)
        except ValueError:
            pass
    print("  %-15s 接受: %s" % (s, accepted))
print("  注意：liblinear + penalty=None 能过 _check_solver，但会在魔数表报错")

== 被忽略 / 被警告 / 被报错的参数 ==
  intercept_scaling(liblinear)  |dcoef|=6.38e-04
  intercept_scaling(lbfgs)      |dcoef|=0.00e+00  （静默忽略）
  l1_ratio + l2                 -> ["l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)"]
  warm_start(liblinear) 二次拟合 |dcoef|=0.00e+00  （静默忽略）
  n_jobs=2 + liblinear          -> ["'n_jobs' > 1 does not have any effect when 'solver' is set to 'libline"]

== _check_solver 能提前拦下的 penalty ==
  lbfgs           接受: ['l2', None]
  liblinear       接受: ['l2', 'l1', None]
  newton-cg       接受: ['l2', None]
  newton-cholesky 接受: ['l2', None]
  sag             接受: ['l2', None]
  saga            接受: ['l2', 'l1', 'elasticnet', None]
  注意：liblinear + penalty=None 能过 _check_solver，但会在魔数表报错


## 9. 概率输出与 `> 0` 阈值

`predict` 的判据是严格 `scores > 0`（`_base.py`），不是 `>=`。决策值恰为 0 时判为 `classes_[0]`。

`predict_proba` 有两条完全不同的实现：

- **OvR**：对每个类做 `expit`，再按行归一化。没有零和防护 —— 如果所有决策值都极负，`expit` 全部下溢到 0，归一化会得到 `0/0`，产生 `NaN` 和 `RuntimeWarning`。
- **multinomial**：`np.c_[-decision, decision]` 后做 softmax；二分类只有 1D 决策值，靠这一步补成两类。

正常数据不会触发 `NaN`，但它是实现里真实存在的边界：OvR 概率不是「各自独立的概率」，而是「强行归一化后的相对分数」。

In [19]:
print("== 决策值恰好为 0 / 极小正负 ==")
m = LogisticRegression(solver="lbfgs").fit(Xs, y)
d0 = m.decision_function(Xs[:1])[0]
w, b = m.coef_[0], m.intercept_[0]
for target in [0.0, 1e-12, -1e-12]:
    x = Xs[0] + (target - d0) * w / (w @ w)
    d = m.decision_function(x[np.newaxis, :])[0]
    print("  目标 %.0e -> 决策 %+.3e -> predict %d"
          % (target, d, m.predict(x[np.newaxis, :])[0]))
print("  classes_ =", m.classes_)

== 决策值恰好为 0 / 极小正负 ==
  目标 0e+00 -> 决策 -1.249e-16 -> predict 0
  目标 1e-12 -> 决策 +9.999e-13 -> predict 1
  目标 -1e-12 -> 决策 -9.999e-13 -> predict 0
  classes_ = [0 1]


In [20]:
print("== OvR 概率的边界：0/0 ==")
m2 = LogisticRegression(solver="lbfgs").fit(Xs, y)
m2.decision_function = lambda Z: np.full((len(Z), 2), -800.0)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    p = m2.predict_proba(Xs[:3])
print("  predict_proba =")
print(p)
print("  出现 NaN:", np.isnan(p).any())
print("  警告:", [str(w.message) for w in caught])

== OvR 概率的边界：0/0 ==
  predict_proba =
[[nan nan]
 [nan nan]
 [nan nan]]
  出现 NaN: True
  警告: ['invalid value encountered in divide']


## 10. 三份 log loss、dtype 与 `n_threads`

同一个 log loss 在库里有三份独立实现：

| 使用它的 solver | 实现 | 能否读源码 |
|---|---|---|
| `lbfgs` / `newton-cg` / `newton-cholesky` | `_loss/_loss*.so`（`HalfBinomialLoss` / `HalfMultinomialLoss`） | 否 |
| `sag` / `saga` | `_sag_fast*.so` 内部的 log / multinomial loss | 否 |
| `liblinear` | C++ 内部的 logistic 损失 | 否 |

三者的数学形式相同，但**归一化方式不同**：`LinearModelLoss` 返回 `sum_i sw_i * loss_i`，SAG 的目标是 `(1/n) sum_i loss_i`，liblinear 是 `C * sum_i loss_i + 0.5||w||^2`。这也是同一个 `C` 需要三种缩放的原因。

**dtype 能力矩阵**：`lbfgs` 强制 `float64`；其余 solver 接受 `[float64, float32]`，但 `liblinear` 内部再转回 `float64`。所以 `float32` 输入下，只有 `newton-cg` / `newton-cholesky` / `sag` / `saga` 的 `coef_` 保持 `float32`。

**`n_threads`**：`_logistic_regression_path` 里有一段看起来很讲究的分支，注释说「未来希望用 OpenMP 线程」，但 1.3.2 的两个分支都赋值 `1` —— 实际并行仍然靠 joblib 按类分发，损失内核是单线程。

In [21]:
print("== 三份实现的位置 ==")
print("  lbfgs / newton-cg / newton-cholesky ->", M_LOSS_SO.__name__,
      "->", os.path.basename(M_LOSS_SO.__file__))
print("  sag / saga                          ->", M_SAG_FAST.__name__,
      "->", os.path.basename(M_SAG_FAST.__file__))
print("  liblinear                           ->", M_LIBLINEAR.__name__,
      "->", os.path.basename(M_LIBLINEAR.__file__))
print()

X32 = Xs.astype(np.float32)
y32 = y.astype(np.float32)
m32 = LogisticRegression(solver="newton-cg").fit(X32, y32)
w32 = np.concatenate([m32.coef_.ravel(), m32.intercept_]).astype(np.float32)
w64 = w32.astype(np.float64)

v32 = LinearModelLoss(base_loss=HalfBinomialLoss(), fit_intercept=True).loss(
    w32, X32, y32, np.ones(len(y), dtype=np.float32), 1.0)
v64 = LinearModelLoss(base_loss=HalfBinomialLoss(), fit_intercept=True).loss(
    w64, Xs, y.astype(np.float64), np.ones(len(y), dtype=np.float64), 1.0)
print("== 同一组系数，两种 dtype ==")
print("  float32 loss = %.9f" % v32)
print("  float64 loss = %.9f" % v64)
print("  差值         = %.2e" % abs(float(v32) - float(v64)))
print()

print("== float32 输入下 coef_ 的 dtype ==")
for s in solvers:
    m = LogisticRegression(solver=s, max_iter=300, random_state=0).fit(X32, y32)
    print("  %-15s coef_.dtype=%-8s n_iter_=%s"
          % (s, m.coef_.dtype, m.n_iter_.tolist()))

== 三份实现的位置 ==
  lbfgs / newton-cg / newton-cholesky -> sklearn._loss._loss -> _loss.cpython-38-x86_64-linux-gnu.so
  sag / saga                          -> sklearn.linear_model._sag_fast -> _sag_fast.cpython-38-x86_64-linux-gnu.so
  liblinear                           -> sklearn.svm._liblinear -> _liblinear.cpython-38-x86_64-linux-gnu.so

== 同一组系数，两种 dtype ==
  float32 loss = 111.928733230
  float64 loss = 111.928737089
  差值         = 3.86e-06

== float32 输入下 coef_ 的 dtype ==
  lbfgs           coef_.dtype=float64  n_iter_=[10]
  liblinear       coef_.dtype=float64  n_iter_=[5]
  newton-cg       coef_.dtype=float32  n_iter_=[9]
  newton-cholesky coef_.dtype=float32  n_iter_=[5]
  sag             coef_.dtype=float32  n_iter_=[26]
  saga            coef_.dtype=float32  n_iter_=[16]


In [22]:
show_lines([(1292, 1302)], title="== n_threads 的赋值：两个分支都等于 1 ==")

from sklearn.utils._openmp_helpers import _openmp_effective_n_threads
print("  OpenMP 可用线程数:", _openmp_effective_n_threads())
print("  但 LogisticRegression 的损失内核没有使用它")

== n_threads 的赋值：两个分支都等于 1 ==
  L1292-L1302
1292|         if (
1293|             solver in ["lbfgs", "newton-cg", "newton-cholesky"]
1294|             and len(classes_) == 1
1295|             and effective_n_jobs(self.n_jobs) == 1
1296|         ):
1297|             # In the future, we would like n_threads = _openmp_effective_n_threads()
1298|             # For the time being, we just do
1299|             n_threads = 1
1300|         else:
1301|             n_threads = 1
1302| 

  OpenMP 可用线程数: 10
  但 LogisticRegression 的损失内核没有使用它


## 11. 小结：一张「参数 → 生效位置」表

| 参数 | 生效位置 | 实际作用 / 陷阱 |
|---|---|---|
| `solver` | `fit` 早退 / path 四支 | 6 个字符串对应 5 套后端，决定可读性、penalty、dtype、`n_iter_` 语义 |
| `C` | path 的 `1/C`、`1/(C·sw_sum)`、`alpha/n`、liblinear 的 `C` | 进后端前被换成四种尺度；`penalty=None` 下失效 |
| `penalty` | `_check_solver` + 魔数表 + SAG 的 `alpha/beta` | `l1` 只有 liblinear / saga；`elasticnet` 只有 saga |
| `dual` | `_check_solver` + 魔数表 | 只有 liblinear + `l2` 支持 |
| `multi_class` | `_check_multi_class` + `classes_=[None]` | `auto` 是能力表；liblinear / cholesky 强制 OvR；multinomial 共用一次 joblib 迭代 |
| `class_weight` | 折叠进 `sample_weight` | `balanced` 在 OvR 掩码之前按全量标签计算 |
| `sample_weight` | `_check_sample_weight` + newton-cholesky 归一化 | 与 `C` 耦合：整体缩放权重 = 缩放 `C` |
| `fit_intercept` | `w0` 多一维，末尾拆出 `intercept_` | liblinear 用合成特征，截距同样被正则 |
| `intercept_scaling` | 仅 liblinear | 其余 solver 静默忽略 |
| `n_jobs` | `Parallel(...)` | OvR 走进程或线程；liblinear 无效并警告 |
| `warm_start` | 传给 path 的 `coef` | liblinear 早退，静默无效 |
| `tol` | 各后端 | 量纲不同，不能跨 solver 比较 |
| `max_iter` | 各后端 | liblinear 每类计数；sag 是 epoch；牛顿法是外层迭代 |
| `random_state` | liblinear / sag / saga | lbfgs 与两种牛顿法完全确定 |
| `n_threads`（内部） | path | 1.3.2 两个分支都写 1，OpenMP 未启用 |

一句话收尾：**读 `LogisticRegression` 的源码，90% 的代码在做「把统一的参数翻译成五种后端的方言」；真正的算法在 `.so` 和 scipy 里。**